### **Youtube vedio ChatBot using RAG: -**

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
import os
from youtube_transcript_api import (
    YouTubeTranscriptApi,
    NoTranscriptFound,
    TranscriptsDisabled
)
from urllib.parse import urlparse, parse_qs
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

API_KEY = os.getenv("GOOGLE_API_KEY")
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=API_KEY
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5525.91it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


## 1. **Indexing : -**
### 1.1 Load the Youtube Transcript : -

In [14]:
def extract_video_id(url: str):
    try:
        parsed = urlparse(url)
        domain = parsed.netloc.lower()

        valid_domains = {
            "youtube.com",
            "www.youtube.com",
            "m.youtube.com",
            "youtu.be",
            "www.youtu.be",
        }

        if domain not in valid_domains:
            print("❌ Invalid URL: This is not a YouTube link.")
            return None

        # https://youtu.be/<id>
        if "youtu.be" in domain:
            video_id = parsed.path.strip("/")

        # https://youtube.com/watch?v=<id>
        elif parsed.path == "/watch":
            video_id = parse_qs(parsed.query).get("v", [None])[0]

        # https://youtube.com/embed/<id>
        elif parsed.path.startswith("/embed/"):
            video_id = parsed.path.split("/")[2]

        # https://youtube.com/shorts/<id>
        elif parsed.path.startswith("/shorts/"):
            video_id = parsed.path.split("/")[2]

        else:
            video_id = None

        if not video_id:
            print("❌ Invalid YouTube video URL.")
            return None

        return video_id

    except Exception:
        print("❌ Invalid URL format.")
        return None


url = "https://youtu.be/OD0R2q97wRw?si=ZvcKQRY0P_6lS4Oc"

video_id = extract_video_id(url)

if video_id:
    id = video_id
    print(id)

OD0R2q97wRw


In [15]:
def get_english_transcript(video_id):
    api = YouTubeTranscriptApi()

    try:
        transcript_list = api.list(video_id)

        # Prefer English
        try:
            transcript = transcript_list.find_transcript(
                ["en", "en-US", "en-GB"]
            )
            print(f"Using English transcript ({transcript.language})")

        except NoTranscriptFound:
            transcript = next(iter(transcript_list))

            print(
                f"Original language: {transcript.language} "
                f"({transcript.language_code})"
            )

            if transcript.is_translatable:
                print("Translating to English...")
                transcript = transcript.translate("en")
            else:
                print("Translation unavailable. Using original transcript.")

        return " ".join(
            item.text for item in transcript.fetch()
        )

    except TranscriptsDisabled:
        print("Transcripts are disabled.")
        return None

    except NoTranscriptFound:
        print("No transcript found.")
        return None


text = get_english_transcript(id)

if text:
    print(text)

Using English transcript (English (auto-generated))
this is the entire one piece timeline it spans for over 5,000 years and includes every important event that has ever happened in the story and today I'll explain the entire thing from start to finish in chological order so it's very simple to follow along throughout the video there will also be a timeline animation at the bottom of the screen so you can easily keep track of exactly when are certain events taking place in order to start from the very beginning we need to go back around 5,000 years when Oh's great tree of omniscience was planted in the ground and became the foundation for the library then a thousand years after that alabara Palace was constructed and thus began the kingdom of alabasta it was established when a person named cahita conquered Sandy Island a calendar starting at year Zer was established 1,524 years before the main story which is going to make keeping track of everything much simpler from now on the next thi

In [ ]:
# video_id = "3Gmo0EXHyKg"  
# try:
#     transcript = YouTubeTranscriptApi().fetch(video_id)

#     text = " ".join([item.text for item in transcript])
#     print(text)

# except TranscriptsDisabled:
#     print("Transcripts are disabled for this video.")

### 1.2 .Devide the transcript into chunks

In [16]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=200)
chunks = splitter.create_documents([text])
print(len(chunks))
chunks

86


[Document(metadata={}, page_content="this is the entire one piece timeline it spans for over 5,000 years and includes every important event that has ever happened in the story and today I'll explain the entire thing from start to finish in chological order so it's very simple to follow along throughout the video there will also be a timeline animation at the bottom of the screen so you can easily keep track of exactly when are certain events taking place in order to start from the very beginning we need to go back around 5,000 years when Oh's great tree of omniscience was planted in the ground and became the foundation for the library then a thousand years after that alabara Palace was constructed and thus began the kingdom of alabasta it was established when a person named cahita conquered Sandy Island a calendar starting at year Zer was established 1,524 years before the main story which is going to make keeping track of everything much simpler from now on the next thing we know is t

### 1.3 Convert it into vector and stored in the vecotr store :-

In [17]:
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
vector_store.index_to_docstore_id

{0: 'b7757690-1e8b-438d-aadf-bcd66b37dd96',
 1: 'b945733f-f39e-4803-b0a2-b6563e544278',
 2: '0a0d8900-7614-4d51-b48d-7bafbf52a56c',
 3: 'e1a635ae-cb6b-454f-827f-f9a30204b99b',
 4: 'e9f380f8-143e-4e20-8201-36d03fb7b09c',
 5: '20abc799-48f1-4173-9263-f84e7cc5ba4e',
 6: 'bcf4ac08-6cde-4dbb-8306-8f45821dd3d7',
 7: 'e4e1bcc1-009b-448b-a54d-439cb0a023a1',
 8: '97dbb694-6d42-4f5c-9349-d751d331009c',
 9: '9d520853-7ee8-4f04-9f41-2ca0146cc423',
 10: '942e8c2e-f246-4a79-9295-10da5ed3c81b',
 11: '517ea97e-1dee-4924-a503-259061b808fb',
 12: 'e5fcfe46-9e51-41e8-bfa2-2ff422c65a52',
 13: 'f32604a4-51f7-4f75-b497-1cf6b7aba878',
 14: '96b295bc-3fd5-4503-bae4-870059076ed6',
 15: '4f3c27e1-e584-4657-aeef-3e0867d82d59',
 16: '53d0f4bd-4a80-4e21-b2e3-ec6588f5e2f3',
 17: '67839c12-141b-4809-94f0-6e09b1ccf2ed',
 18: '948ce5ae-4413-46ad-be3c-138048c2b5a5',
 19: '206bc0c8-c28c-4248-b710-971c56067e87',
 20: '0c9c81b0-7a17-41ee-b3d2-611025d5f436',
 21: '20d21d7d-8b6b-40c3-a0ca-3449b467a2f9',
 22: '33183ff7-7ee3-

## **2. Retrival :-**

### Retrieve the related content 

In [40]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 6,
        "fetch_k": 10,
        "lambda_mult": 0.9
    }
)
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x16b452cf0>, search_type='mmr', search_kwargs={'k': 6, 'fetch_k': 10, 'lambda_mult': 0.9})

In [19]:
retriever.invoke("Who is joyboy")

[Document(id='e9f380f8-143e-4e20-8201-36d03fb7b09c', metadata={}, page_content="after berries became a thing the era of slaves began the slaves in the oppress came to idolize and prayed to the Sun God who they believed would come to liberate them from their oppressors somewhere around this time a person called Joy boy who wore a straw hat set out to see and became the first pirate at some point he also ate the devil fruit called the hitoo noi model naika during his travels Joy boy befriended zunes and The Iron Giant EMT back then fishmen Island was the only place where fishmen gathered in order to bask in the sunlight that came from the Sun tree Eve it was a settlement they established under the sea to avoid the persecution they faced on land Joy boy visited the island in proed he'd raise the giant ship Noah to the surface a promise he was not able to keep for an unknown reason during his journeys Joy boy arrived on a certain island in the new world which would later be known as laugh 

## **3 .Augmentation : -**

### Create the the Prompt

In [ ]:
# prompt = PromptTemplate(
#     template="""
#       You are a helpful assistant.
#       Answer ONLY from the provided transcript context.
#       If the context is insufficient, just say you don't know.

#       {context}
#       Question: {question}
#     """,
#     input_variables = ['context', 'question']
# )


In [43]:
prompt = PromptTemplate(
    template="""
You are a helpful AI assistant.

Your primary source of information is the provided YouTube transcript.

Rules:
1. Answer questions using the transcript whenever the answer is available.
2. If the user explicitly asks for information beyond the video (for example: "explain beyond this video", "what else do you know", "tell me more", "latest information", "real-world examples", "expand on this topic", etc.), you may use your general knowledge. Clearly mention that the additional information is **not from the video transcript**.
3. If the transcript does not contain the answer and the user did NOT ask for information beyond the video, respond:
   "I couldn't find that information in the provided video transcript."
4. Never invent or assume details that are not supported by the transcript when answering transcript-based questions.
5. When your answer combines transcript information with general knowledge, clearly separate the two sections.

Transcript:
{context}

Question:
{question}

Answer:
""",
    input_variables=["context", "question"],
)

In [20]:
question          = "Who is joy boy ?"
retrieved_docs    = retriever.invoke(question)
retrieved_docs

[Document(id='e9f380f8-143e-4e20-8201-36d03fb7b09c', metadata={}, page_content="after berries became a thing the era of slaves began the slaves in the oppress came to idolize and prayed to the Sun God who they believed would come to liberate them from their oppressors somewhere around this time a person called Joy boy who wore a straw hat set out to see and became the first pirate at some point he also ate the devil fruit called the hitoo noi model naika during his travels Joy boy befriended zunes and The Iron Giant EMT back then fishmen Island was the only place where fishmen gathered in order to bask in the sunlight that came from the Sun tree Eve it was a settlement they established under the sea to avoid the persecution they faced on land Joy boy visited the island in proed he'd raise the giant ship Noah to the surface a promise he was not able to keep for an unknown reason during his journeys Joy boy arrived on a certain island in the new world which would later be known as laugh 

In [21]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"after berries became a thing the era of slaves began the slaves in the oppress came to idolize and prayed to the Sun God who they believed would come to liberate them from their oppressors somewhere around this time a person called Joy boy who wore a straw hat set out to see and became the first pirate at some point he also ate the devil fruit called the hitoo noi model naika during his travels Joy boy befriended zunes and The Iron Giant EMT back then fishmen Island was the only place where fishmen gathered in order to bask in the sunlight that came from the Sun tree Eve it was a settlement they established under the sea to avoid the persecution they faced on land Joy boy visited the island in proed he'd raise the giant ship Noah to the surface a promise he was not able to keep for an unknown reason during his journeys Joy boy arrived on a certain island in the new world which would later be known as laugh Tale on it Joy booy hit his treasure although we don't know what it is it was d

In [24]:
final_prompt = prompt.invoke({"context": context_text, "question": question})
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      after berries became a thing the era of slaves began the slaves in the oppress came to idolize and prayed to the Sun God who they believed would come to liberate them from their oppressors somewhere around this time a person called Joy boy who wore a straw hat set out to see and became the first pirate at some point he also ate the devil fruit called the hitoo noi model naika during his travels Joy boy befriended zunes and The Iron Giant EMT back then fishmen Island was the only place where fishmen gathered in order to bask in the sunlight that came from the Sun tree Eve it was a settlement they established under the sea to avoid the persecution they faced on land Joy boy visited the island in proed he'd raise the giant ship Noah to the surface a promise he was not able to keep for an unknown reaso

### **4.Generation : -**

In [25]:
result = model.invoke(final_prompt)
print(result.content)

Joy boy was a person who wore a straw hat and set out to sea, becoming the first pirate. He ate the devil fruit called the Hito Hito no Mi, Model: Nika. He befriended Zunesha and The Iron Giant EMT. He visited Fishman Island and promised to raise the giant ship Noah to the surface, a promise he was not able to keep. During his travels, he arrived on an island in the New World, later known as Laugh Tale, where he hid his treasure, described as something that could completely turn the world upside down.

He had a citizen of Wano write the poneglyphs for him, with one outlining his apology to the Fish-Men for not fulfilling his promise and promising that someone would fulfill it in the future. He also left a message at an unnamed island, claiming he would return in 800 years. Later, 20 kingdoms around the world banded together and waged war against Joy boy in an effort to kill him and destroy the ancient Kingdom. Before his demise, Joy boy's allies managed to hide the ancient weapons, and

## **Now make a full chain : -**

In [26]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()

In [27]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [28]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [29]:
parallel_chain.invoke('what the story is all about ?')


{'context': "this is the entire one piece timeline it spans for over 5,000 years and includes every important event that has ever happened in the story and today I'll explain the entire thing from start to finish in chological order so it's very simple to follow along throughout the video there will also be a timeline animation at the bottom of the screen so you can easily keep track of exactly when are certain events taking place in order to start from the very beginning we need to go back around 5,000 years when Oh's great tree of omniscience was planted in the ground and became the foundation for the library then a thousand years after that alabara Palace was constructed and thus began the kingdom of alabasta it was established when a person named cahita conquered Sandy Island a calendar starting at year Zer was established 1,524 years before the main story which is going to make keeping track of everything much simpler from now on the next thing we know is that around year 400 the 

In [30]:
main_chain = parallel_chain | prompt | model | parser

In [44]:
result = main_chain.invoke('what the story is all about ?')
print(result)


I don't know. The transcript provides a chronological timeline of events within the One Piece story, but it does not explain what the story is all about in terms of its overarching plot or main objective.
